***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.1 幅度、相位与成像关系](5_1_spatial_frequencies.ipynb)
    * 下一节：[5.3 网格化与反网格化](5_3_gridding_and_degridding.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import HTML
import track_simulator
import westerbork_92m_constants as wsrt92
import jvla_d_constants as jvla_d
import jvla_a_constants as jvla_a
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


In [ ]:
HTML('../style/code_toggle.html')


## 5.2 采样函数与点扩散函数<a id='imaging:sec:samplingPSF'></a>

第 5.1 节说明了复可见度如何作为条纹系数参与图像合成。真实干涉阵不能测得完整、连续的可见度函数 $V(u,v)$，只能获得有限数量的离散样本。因此，成像时进行逆傅里叶变换的不是 $V(u,v)$ 本身，而是经过采样函数截断和加权后的可见度分布。脏图像、脏波束以及后续的去卷积问题均由这种不完备采样产生。

采样在频域中表现为乘法，在图像域中对应卷积；这一关系是解释脏图像和脏波束的基础。本节先讨论理想化的规则采样和圆形孔径，再转向真实阵列的 $uv$ 覆盖与点扩散函数，最后说明观测时长、带宽、阵列布局和权重如何共同决定空间尺度响应。


In [ ]:
FIG_DIR = Path('figures')
FIG_DIR.mkdir(exist_ok=True)
C = 299792458.0
KAT7_ENU = np.array([
    [ 25.095,  -9.095,  0.045],
    [ 90.284,  26.380, -0.226],
    [  3.985,  26.893,  0.000],
    [-21.605,  25.494,  0.019],
    [-38.272,  -2.592,  0.391],
    [-61.595, -79.699,  0.702],
    [-87.988,  75.754,  0.138],
], dtype=float)
KAT7_LATITUDE = -(30.0 + 43.0 / 60.0 + 17.34 / 3600.0)


def rgb_to_gray(img):
    arr = img[..., :3].astype(float)
    if arr.max() > 1.5:
        arr /= 255.0
    return 0.2989 * arr[..., 0] + 0.5870 * arr[..., 1] + 0.1140 * arr[..., 2]


def normalize(arr):
    arr = np.asarray(arr, dtype=float)
    maxval = np.max(np.abs(arr))
    return arr if maxval == 0 else arr / maxval


def circular_mask(size, outer_radius, inner_radius=0.0):
    yy, xx = np.mgrid[0:size, 0:size]
    rr = np.sqrt((xx - size / 2.0) ** 2 + (yy - size / 2.0) ** 2)
    mask = ((rr <= outer_radius) & (rr >= inner_radius)).astype(float)
    return mask


def fft_image(image):
    return np.fft.fftshift(np.fft.fft2(image))


def ifft_image(vis):
    return np.fft.ifft2(np.fft.ifftshift(vis)).real


def dirty_image_from_mask(image, mask):
    return ifft_image(fft_image(image) * mask)


def dirty_beam_from_mask(mask):
    beam = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(mask))).real
    return normalize(beam)


def central_crop(image, half_width=56):
    cy = image.shape[0] // 2
    cx = image.shape[1] // 2
    return image[cy - half_width:cy + half_width, cx - half_width:cx + half_width]


def simulate_uv_tracks(enu, latitude, declination, obs_hours, int_minutes=2.0, freqs_hz=(1.4e9,)):
    uvw_m = track_simulator.sim_uv(
        -7.5 * obs_hours, declination, obs_hours, int_minutes / 60.0, enu, latitude,
        include_autocorrelations=False,
    )
    uv_list = []
    for freq in np.atleast_1d(freqs_hz):
        wavelength = C / freq
        uv = uvw_m[:, :2] / wavelength
        uv_list.append(uv)
        uv_list.append(-uv)
    return np.vstack(uv_list)


def rasterize_uv(uv, grid_size=384, uv_max=None, binary=False):
    uv = np.asarray(uv, dtype=float)
    if uv_max is None:
        uv_max = np.max(np.abs(uv)) * 1.02
    mask = np.zeros((grid_size, grid_size), dtype=float)
    if uv_max == 0:
        return mask
    x = np.clip(np.round((uv[:, 0] / uv_max * 0.5 + 0.5) * (grid_size - 1)).astype(int), 0, grid_size - 1)
    y = np.clip(np.round((uv[:, 1] / uv_max * 0.5 + 0.5) * (grid_size - 1)).astype(int), 0, grid_size - 1)
    for xi, yi in zip(x, y):
        if binary:
            mask[yi, xi] = 1.0
        else:
            mask[yi, xi] += 1.0
    return mask


def synthetic_sky(size=256):
    y, x = np.mgrid[-1:1:complex(size), -1:1:complex(size)]
    sky = (
        0.9 * np.exp(-0.5 * ((x / 0.45) ** 2 + (y / 0.22) ** 2))
        + 0.35 * np.exp(-0.5 * (((x + 0.35) / 0.08) ** 2 + ((y - 0.25) / 0.06) ** 2))
        + 0.25 * np.exp(-0.5 * (((x - 0.32) / 0.05) ** 2 + ((y + 0.18) / 0.05) ** 2))
    )
    return normalize(sky)


### 5.2.1 采样方程与脏图像

若理想可见度函数记为 $V(u,v)$，真实阵列的离散采样可写成一个加权采样函数

$$
S(u,v)=\sum_{i=1}^{Q} w_i\,\delta(u-u_i)\,\delta(v-v_i),
$$

于是加权后的离散可见度分布是

$$
V_s(u,v)=S(u,v)\,V(u,v).
$$

这里采用[章首约定](5_0_introduction.ipynb#imaging:sec:normalization_units)，将 $w_i=w_i^{\rm stat}q_i$ 分为逆噪声方差权重和成像重加权；标记为无效的数据不进入求和。$V_s$ 是由狄拉克 delta 函数组成的加权分布，并不表示仪器在所有 $(u,v)$ 处连续测得了 $V$。对于直接离散傅里叶变换（DFT），令 $\eta=\sum_i w_i$，并将脏波束中心响应归一化为 1，可得

$$
I_D(l,m)=\frac{1}{\eta}\mathcal{F}^{-1}\{V_s\}=\frac{1}{\eta}\mathcal{F}^{-1}\{S\}\,*\,\mathcal{F}^{-1}\{V\}
       = B_D(l,m) * I_{\rm app}(l,m),
$$

其中

$$
B_D(l,m)=\frac{1}{\eta}\mathcal{F}^{-1}\{S(u,v)\},\qquad B_D(0,0)=1
$$

称为脏波束或点扩散函数，$I_D$ 则称为脏图像。这是二维、线性且平移不变近似下的卷积方程；宽视场中的方向相关响应会使 PSF 随位置变化，此时不能继续使用单一的全局卷积。对于实值 Stokes-I 天空，将每个 $(u_i,v_i,V_i)$ 与 $(-u_i,-v_i,V_i^*)$ 成对表示可保证脏图为实值；若数据本身已经同时存有两种相关顺序，则不能再无条件复制。公式中的卷积表示算子关系，最终单位是 $\mathrm{Jy\,dirty\ beam^{-1}}$、$\mathrm{Jy\,pixel^{-1}}$ 还是其他形式，取决于章首所列的 PSF 和像元归一化约定。

规则采样是上述一般公式的一种特殊情形。若在一维中使用梳状函数（Shah function）

$$
\operatorname{III}_a(x)=\sum_{n=-\infty}^{\infty}\delta(x-na)
$$

去采样连续信号，就会得到熟悉的 Nyquist 条件和混叠现象。射电干涉测量与普通数字化的关键差别在于：CCD 在图像域采样，而干涉阵在空间频率域采样，因此混叠和卷积伪影会出现在完全不同的位置。


In [ ]:
x = np.linspace(-5.0, 5.0, 2001)
comb_positions = np.arange(-5, 6, 1)
low_freq = np.sin(1.2 * x)
high_freq = 0.5 * np.sin(5.0 * x)

sampled_low = np.sin(1.2 * comb_positions)
sampled_high = 0.5 * np.sin(5.0 * comb_positions)

fig, axes = plt.subplots(3, 1, figsize=(9.0, 8.0), sharex=True)
axes[0].vlines(comb_positions, 0.0, 1.0, color='#1d3557', lw=2)
axes[0].set_ylim(0.0, 1.1)
axes[0].set_title('Shah sampling function')
axes[0].grid(alpha=0.2)

axes[1].plot(x, low_freq, lw=2, label='low spatial frequency')
axes[1].plot(x, high_freq, lw=2, label='high spatial frequency')
axes[1].set_title('continuous signals')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.2)

axes[2].plot(comb_positions, sampled_low, 'o-', lw=2, label='adequately sampled')
axes[2].plot(comb_positions, sampled_high, 'o-', lw=2, label='aliased after sampling')
axes[2].set_title('sampled values')
axes[2].set_xlabel('position')
axes[2].legend(frameon=False)
axes[2].grid(alpha=0.2)

fig.tight_layout()
fig.savefig(FIG_DIR / 'sampling_aliasing_1d.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![规则采样与混叠](figures/sampling_aliasing_1d.png)

**图 5.2.1：一维梳状函数采样与混叠。** 低空间频率信号在采样后仍能保持原有变化趋势；高空间频率信号则在离散样本上表现为较低频率的变化，即发生混叠。干涉成像虽然在频域而不是图像域中采样，但缺失或混叠的频率同样会系统性地影响恢复结果。


### 5.2.2 圆盘采样、环形采样与空间滤波

采样函数最直观的物理原型，是一个有限口径。若在频域中保留一个圆盘区域，实际上做的是低通滤波：保留大尺度结构、抹去高频细节。若只保留一个环形区域，则相当于带通滤波：既丢掉最低频，也丢掉最高频，只保留中间尺度结构。

这个直觉对于干涉阵尤其重要。缺少长基线意味着高频不够，图像会变模糊；缺少短基线意味着低频不够，图像中的大尺度背景会被削弱甚至整体被“减去平均值”。因此，常说的“短基线决定大尺度结构、长基线决定分辨率”，本质上就是对采样函数几何形状的另一种说法。


In [ ]:
galaxy = rgb_to_gray(mpimg.imread('figures/synthetic_spiral_galaxy.png'))[::2, ::2]
size = galaxy.shape[0]
disk_mask = circular_mask(size, 26, 0)
annulus_mask = circular_mask(size, 70, 18)

disk_beam = dirty_beam_from_mask(disk_mask)
annulus_beam = dirty_beam_from_mask(annulus_mask)
disk_image = normalize(dirty_image_from_mask(galaxy, disk_mask))
annulus_image = normalize(dirty_image_from_mask(galaxy, annulus_mask))

fig, axes = plt.subplots(2, 3, figsize=(12.5, 8.0))
rows = [
    ('disk mask (low-pass)', disk_mask, central_crop(disk_beam, 52), disk_image),
    ('annulus mask (band-pass)', annulus_mask, central_crop(annulus_beam, 52), annulus_image),
]
for row, (title, mask, beam, image) in enumerate(rows):
    axes[row, 0].imshow(mask, cmap='gray', origin='lower')
    axes[row, 0].set_title(title)
    axes[row, 1].imshow(beam, cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    axes[row, 1].set_title('PSF')
    axes[row, 2].imshow(image, cmap='gray', origin='lower')
    axes[row, 2].set_title('filtered image')
    for ax in axes[row]:
        ax.set_xticks([])
        ax.set_yticks([])

fig.tight_layout()
fig.savefig(FIG_DIR / 'sampling_aperture_filters.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![圆盘与环形采样](figures/sampling_aperture_filters.png)

**图 5.2.2：圆盘采样与环形采样的空间滤波作用。** 完整圆盘主要保留低频，因此图像趋于平滑；环形采样会抑制中心低频分量，使大尺度亮度背景减弱，并保留较多边缘和中等尺度结构。对于干涉阵，缺失短基线造成的大尺度通量损失正是这种空间滤波效应的表现。


### 5.2.3 综合孔径时间对脏波束的影响

真实阵列的采样函数不再是规则几何图形，而是由阵列布局和地球自转共同决定。随着观测时间增加，同一组物理基线在 $uv$ 平面上形成逐渐延长的轨迹，原本稀疏的离散采样逐步扩展为弧线或椭圆。覆盖通常会变得更加均匀，较强旁瓣可能降低；主瓣宽度则由实际采得的最大投影基线和权重共同决定，并不一定随观测时间单调减小。


In [ ]:
time_cases = [('10 min', 10.0 / 60.0), ('2 h', 2.0), ('12 h', 12.0)]
time_uv = [simulate_uv_tracks(KAT7_ENU, KAT7_LATITUDE, -30.0, hours, int_minutes=1.0) for _, hours in time_cases]
uv_max = max(np.max(np.abs(uv)) for uv in time_uv)

fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.2))
for col, ((label, _), uv) in enumerate(zip(time_cases, time_uv)):
    axes[0, col].scatter(uv[:, 0] / 1e3, uv[:, 1] / 1e3, s=1.2, c='#1d3557', alpha=0.45)
    axes[0, col].set_title(label)
    axes[0, col].set_xlabel(r'$u$ [$k\lambda$]')
    if col == 0:
        axes[0, col].set_ylabel(r'$v$ [$k\lambda$]')
    axes[0, col].grid(alpha=0.2)
    lim = uv_max / 1e3 * 1.05
    axes[0, col].set_xlim(-lim, lim)
    axes[0, col].set_ylim(-lim, lim)
    axes[0, col].set_aspect('equal')

    mask = rasterize_uv(uv, grid_size=384, uv_max=uv_max)
    beam = central_crop(dirty_beam_from_mask(mask), 54)
    axes[1, col].imshow(beam, cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    axes[1, col].set_title('dirty beam')
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.tight_layout()
fig.savefig(FIG_DIR / 'kat7_time_psf_evolution.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![观测时长与脏波束](figures/kat7_time_psf_evolution.png)

**图 5.2.3：观测时长对 $uv$ 覆盖与脏波束的影响。** KAT-7 指向赤纬 $-30^\circ$，并以中天为观测中心，观测时长由 10 分钟增加到 12 小时。观测时间的增加不改变物理基线，但地球自转能够补充更多投影方向，使旁瓣分布通常更加均匀。图中仅使用互相关数据，未将自相关误认为零间距可见度。


### 5.2.4 带宽综合对采样的改善

在单个频率通道内，给定物理基线的空间频率坐标满足 $u=b_x/\lambda$、$v=b_y/\lambda$。因此，当观测覆盖有限带宽，并将多个窄频率通道联合用于成像时，同一条物理基线会在 $uv$ 平面上沿径向形成一组采样点，这是多频综合成像的几何基础。增大分数带宽会增加径向跨度；在固定带宽内进一步细分通道，只会使该轨迹的离散表示更加密集，不会产生新的最短或最长物理基线。宽带成像还必须同时建立天空频谱和频率相关主波束模型，不能将所有通道简单视为同一天空亮度分布。


In [ ]:
band_cases = [
    ('1.40 GHz only', np.array([1.40e9])),
    ('two band edges', np.array([1.35e9, 1.45e9])),
    ('32 channels', np.linspace(1.35e9, 1.45e9, 32)),
]
band_uv = [simulate_uv_tracks(KAT7_ENU, KAT7_LATITUDE, -30.0, 6.0, int_minutes=2.0, freqs_hz=freqs) for _, freqs in band_cases]
uv_max = max(np.max(np.abs(uv)) for uv in band_uv)

fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.2))
for col, ((label, _), uv) in enumerate(zip(band_cases, band_uv)):
    axes[0, col].scatter(uv[:, 0] / 1e3, uv[:, 1] / 1e3, s=0.8, c='#1d3557', alpha=0.35)
    axes[0, col].set_title(label)
    axes[0, col].set_xlabel(r'$u$ [$k\lambda$]')
    if col == 0:
        axes[0, col].set_ylabel(r'$v$ [$k\lambda$]')
    axes[0, col].grid(alpha=0.2)
    lim = uv_max / 1e3 * 1.05
    axes[0, col].set_xlim(-lim, lim)
    axes[0, col].set_ylim(-lim, lim)
    axes[0, col].set_aspect('equal')

    mask = rasterize_uv(uv, grid_size=384, uv_max=uv_max)
    beam = central_crop(dirty_beam_from_mask(mask), 54)
    axes[1, col].imshow(beam, cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    axes[1, col].set_title('dirty beam')
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.tight_layout()
fig.savefig(FIG_DIR / 'kat7_bandwidth_psf_evolution.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![带宽综合与脏波束](figures/kat7_bandwidth_psf_evolution.png)

**图 5.2.4：带宽综合对 $uv$ 采样的影响。** 对 KAT-7 的 6 小时观测，1.35--1.45 GHz 带宽使各物理基线的采样沿径向扩展。两个频带边缘已经确定径向范围，增加中间通道只会在该范围内形成更密集的采样。中心的短基线空缺仍然存在，因为有限频带不能将非零物理间距变为真正的零间距。


### 5.2.5 指向赤纬对点扩散函数的影响

同一阵列观测不同赤纬的目标时，基线在 $uv$ 平面上的投影不同。接近天极时，轨迹更接近同心圆，脏波束也更接近径向对称；接近天赤道时，轨迹变扁甚至退化，脏波束主瓣被拉长，旁瓣增强。因此，PSF 不是阵列本身的固定属性，而是由阵列几何与观测指向共同决定的观测属性。


In [ ]:
dec_cases = [-90.0, -60.0, 0.0, 30.0]
dec_labels = [r'$-90^\circ$', r'$-60^\circ$', r'$0^\circ$', r'$+30^\circ$']
dec_uv = [simulate_uv_tracks(KAT7_ENU, KAT7_LATITUDE, dec, 6.0, int_minutes=2.0) for dec in dec_cases]
uv_max = max(np.max(np.abs(uv)) for uv in dec_uv)

fig, axes = plt.subplots(2, 4, figsize=(15.5, 7.0))
for col, (label, uv) in enumerate(zip(dec_labels, dec_uv)):
    axes[0, col].scatter(uv[:, 0] / 1e3, uv[:, 1] / 1e3, s=1.0, c='#1d3557', alpha=0.4)
    axes[0, col].set_title(label)
    axes[0, col].set_xlabel(r'$u$ [$k\lambda$]')
    if col == 0:
        axes[0, col].set_ylabel(r'$v$ [$k\lambda$]')
    axes[0, col].grid(alpha=0.2)
    lim = uv_max / 1e3 * 1.05
    axes[0, col].set_xlim(-lim, lim)
    axes[0, col].set_ylim(-lim, lim)
    axes[0, col].set_aspect('equal')

    mask = rasterize_uv(uv, grid_size=384, uv_max=uv_max)
    beam = central_crop(dirty_beam_from_mask(mask), 52)
    axes[1, col].imshow(beam, cmap='RdBu_r', origin='lower', vmin=-0.25, vmax=1.0)
    axes[1, col].set_title('dirty beam')
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.tight_layout()
fig.savefig(FIG_DIR / 'kat7_declination_psf_evolution.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![赤纬依赖的脏波束](figures/kat7_declination_psf_evolution.png)

**图 5.2.5：指向赤纬对 $uv$ 覆盖与脏波束的影响。** 同一 KAT-7 阵列指向不同赤纬时，接近南天极的响应更接近圆对称；赤纬升高后，采样轨迹变得更扁，主瓣和旁瓣均表现出明显方向性。因此，实际成像中的分辨率和伪影形态与观测指向相互关联。


### 5.2.6 纯东西向阵列与二维阵列

阵列几何本身也会深刻地塑造采样函数。纯东西向阵列在某些赤纬上能够依靠地球自转补出完整轨迹，但瞬时覆盖通常较差，对低赤纬尤其不利；二维阵列则在短时间内就能给出更多方向上的空间频率样本，因此瞬时 PSF 往往更紧凑，也更适合快变源或快照成像。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
for ax, module, title in [
    (axes[0], wsrt92, 'WSRT 92 m east-west layout (N expanded)'),
    (axes[1], jvla_d, 'JVLA D two-dimensional layout'),
]:
    ax.scatter(module.ENU[:, 0], module.ENU[:, 1], color='#1d3557', s=36)
    ax.set_title(title)
    ax.set_xlabel('E [m]')
    ax.set_ylabel('N [m]')
    ax.grid(alpha=0.25)
    ax.set_aspect('auto' if module is wsrt92 else 'equal')
fig.tight_layout()
fig.savefig(FIG_DIR / 'wsrt_jvla_layouts.png', dpi=180, bbox_inches='tight')
plt.close(fig)

wsrt_fast = simulate_uv_tracks(wsrt92.ENU, wsrt92.ARRAY_LATITUDE, 20.0, 5.0 / 60.0, int_minutes=1.0)
wsrt_long = simulate_uv_tracks(wsrt92.ENU, wsrt92.ARRAY_LATITUDE, 20.0, 6.0, int_minutes=2.0)
jvla_fast = simulate_uv_tracks(jvla_d.ENU, jvla_d.ARRAY_LATITUDE, 20.0, 5.0 / 60.0, int_minutes=1.0)
jvla_long = simulate_uv_tracks(jvla_d.ENU, jvla_d.ARRAY_LATITUDE, 20.0, 6.0, int_minutes=2.0)
uv_max = max(np.max(np.abs(arr)) for arr in [wsrt_fast, wsrt_long, jvla_fast, jvla_long])

fig, axes = plt.subplots(2, 2, figsize=(10.5, 9.0))
items = [
    ('WSRT, 5 min', wsrt_fast),
    ('WSRT, 6 h', wsrt_long),
    ('JVLA D, 5 min', jvla_fast),
    ('JVLA D, 6 h', jvla_long),
]
for ax, (title, uv) in zip(axes.flat, items):
    ax.scatter(uv[:, 0] / 1e3, uv[:, 1] / 1e3, s=0.9, c='#1d3557', alpha=0.35)
    ax.set_title(title)
    ax.set_xlabel(r'$u$ [$k\lambda$]')
    ax.set_ylabel(r'$v$ [$k\lambda$]')
    ax.grid(alpha=0.2)
    lim = uv_max / 1e3 * 1.05
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
fig.tight_layout()
fig.savefig(FIG_DIR / 'wsrt_jvla_uv_comparison.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![WSRT 与 JVLA 布局](figures/wsrt_jvla_layouts.png)

**图 5.2.6：纯东西向阵列与二维阵列的几何布局。** 为显示 WSRT 小于 0.4 m 的南北坐标差，左图相对于约 2.76 km 的东西跨度大幅放大了 $N$ 轴，因而不能依据图中斜率判断真实基线方向。二维布局在地球自转综合之前便具有更多方向的物理基线。

![WSRT 与 JVLA 的瞬时和长时覆盖](figures/wsrt_jvla_uv_comparison.png)

**图 5.2.7：WSRT 与 JVLA D 构型的瞬时和长时 $uv$ 覆盖。** 在相同赤纬下，WSRT 依靠时间综合将近一维的基线分布逐步扩展为二维采样；JVLA 在较短时间内即可获得更丰富的二维覆盖，因此更适于快照成像和低赤纬目标成像。


### 5.2.7 阵列构型本身就是空间滤波器

同一阵列的不同构型，也可以理解为不同的空间滤波器。紧凑构型拥有更多短基线，因此对低空间频率敏感，更善于恢复大尺度扩展辐射；展开构型拥有更多长基线，因此对高空间频率敏感，能够分辨精细结构，但也更容易丢失平滑背景。这不是后处理阶段“选择不同分辨率”的问题，而是采样函数在观测时就已经决定的物理选择。


In [ ]:
sky = synthetic_sky(256)
uv_d = simulate_uv_tracks(jvla_d.ENU, jvla_d.ARRAY_LATITUDE, 30.0, 0.5, int_minutes=1.0)
uv_a = simulate_uv_tracks(jvla_a.ENU, jvla_a.ARRAY_LATITUDE, 30.0, 0.5, int_minutes=1.0)
uv_max = max(np.max(np.abs(uv_d)), np.max(np.abs(uv_a)))
mask_d = rasterize_uv(uv_d, grid_size=256, uv_max=uv_max)
mask_a = rasterize_uv(uv_a, grid_size=256, uv_max=uv_max)
img_d = normalize(dirty_image_from_mask(sky, mask_d))
img_a = normalize(dirty_image_from_mask(sky, mask_a))

fig, axes = plt.subplots(2, 3, figsize=(12.0, 7.4))
axes[0, 0].imshow(sky, cmap='magma', origin='lower')
axes[0, 0].set_title('synthetic sky')
axes[0, 1].imshow(mask_d > 0, cmap='gray', origin='lower')
axes[0, 1].set_title('JVLA D sampling')
axes[0, 2].imshow(mask_a > 0, cmap='gray', origin='lower')
axes[0, 2].set_title('JVLA A sampling')
axes[1, 0].axis('off')
axes[1, 1].imshow(img_d, cmap='magma', origin='lower')
axes[1, 1].set_title('image from D sampling')
axes[1, 2].imshow(img_a, cmap='magma', origin='lower')
axes[1, 2].set_title('image from A sampling')
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig(FIG_DIR / 'jvla_spatial_filtering.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![阵列构型作为空间滤波器](figures/jvla_spatial_filtering.png)

**图 5.2.8：不同 JVLA 构型的空间滤波作用。** 紧凑 D 构型对同一合成天空保留了较强的大尺度背景，但小尺度结构较为模糊；展开 A 构型突出紧致成分，同时削弱平滑扩展辐射。该结果直观表明阵列构型决定了可恢复角尺度的范围。


### 5.2.8 本节小结

本节建立了成像中的基本卷积关系。采样函数在 $uv$ 平面上截断并加权理想可见度，其逆变换在图像域中形成脏波束。因此，观测时长、带宽、指向赤纬、阵列几何和阵列构型都会通过改变采样函数，进而改变脏图像和点扩散函数。

前述示例均将采样函数表示在规则网格上，而真实可见度样本通常并不恰好落在快速傅里叶变换（FFT）的网格点上。为了将不规则采样转换为可高效计算的规则离散傅里叶变换，需要引入网格化与反网格化；相关数值方法将在第 5.3 节讨论。


***

* 下一节：[5.3 网格化与反网格化](5_3_gridding_and_degridding.ipynb)
